In [ ]:
from pathlib import Path
import os, subprocess, sys
src=Path('/kaggle/working/public_audit_source');src.mkdir(exist_ok=True)
for name,source in {'preprocess.py': '"""Turn each DICOM series into a canonical uint8 volume [n_slices, SIZE, SIZE].\n\nCanonical means: fixed physical pixel size, centre crop to a fixed field of view,\nstandard in-plane orientation, standard slice order and a per-series intensity window.\nThe same `load_series` is used for the training cache and for inference.\n\nUsage: python src/preprocess.py [train|test] [n_series_limit]\n"""\nimport os\nimport sys\nfrom multiprocessing import Pool\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport pydicom\n\nDATA = "/mnt/ssd/rsna_knee/data"\nWORK = "/mnt/ssd/rsna_knee/work"\n\nMM_PER_PX = 0.4\nSIZE = 384  # 153.6 mm field of view\nMAX_SLICES = 64\n\n# plane -> (normal axis, slice-order sign, (col axis, sign), (row axis, sign)) in LPS coordinates\n# Sagittal: slices right->left, image columns anterior->posterior, rows superior->inferior\n# Coronal:  slices anterior->posterior, columns patient-right->left, rows superior->inferior\n# Axial:    slices superior->inferior, columns patient-right->left, rows anterior->posterior\nCANON = {\n    0: (+1, (1, +1), (2, -1)),\n    1: (+1, (0, +1), (2, -1)),\n    2: (-1, (0, +1), (1, +1)),\n}\nPLANES = ["Sagittal", "Coronal", "Axial"]\n\n\ndef _read(path):\n    ds = pydicom.dcmread(path)\n    arr = ds.pixel_array.astype(np.float32)\n    if ds.get("PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":\n        arr = arr.max() - arr\n    slope, inter = ds.get("RescaleSlope"), ds.get("RescaleIntercept")\n    if slope is not None and inter is not None:\n        arr = arr * float(slope) + float(inter)\n    iop = [float(v) for v in ds.ImageOrientationPatient] if "ImageOrientationPatient" in ds else None\n    ipp = [float(v) for v in ds.ImagePositionPatient] if "ImagePositionPatient" in ds else None\n    ps = [float(v) for v in ds.PixelSpacing] if "PixelSpacing" in ds else None\n    inst = float(ds.get("InstanceNumber") or 0)\n    frames = list(arr) if arr.ndim == 3 else [arr]\n    return [(f, iop, ipp, ps, inst + i * 1e-3) for i, f in enumerate(frames)]\n\n\ndef _canon_slice(img, iop, ps, plane):\n    """Reorient one slice to the canonical in-plane orientation; returns (img, (row_mm, col_mm))."""\n    ps = ps or [MM_PER_PX * img.shape[0] / SIZE] * 2  # no spacing: assume the image spans the crop\n    if iop is None:\n        return img, ps\n    r, c = np.array(iop[:3]), np.array(iop[3:])\n    _, (cax, csgn), (rax, rsgn) = CANON[plane]\n    if abs(r[cax]) < abs(r[rax]):  # rows and columns are swapped relative to canonical\n        img, r, c, ps = img.T, c, r, ps[::-1]\n    if r[cax] * csgn < 0:\n        img = img[:, ::-1]\n    if c[rax] * rsgn < 0:\n        img = img[::-1]\n    return img, ps\n\n\ndef _resample(img, ps):\n    """Centre crop to SIZE*MM_PER_PX mm, resample to MM_PER_PX, zero-pad to SIZE x SIZE."""\n    out = np.zeros((SIZE, SIZE), np.float32)\n    h, w = img.shape\n    ch = min(h, int(round(SIZE * MM_PER_PX / ps[0])))\n    cw = min(w, int(round(SIZE * MM_PER_PX / ps[1])))\n    y0, x0 = (h - ch) // 2, (w - cw) // 2\n    crop = np.ascontiguousarray(img[y0:y0 + ch, x0:x0 + cw])\n    nh = max(1, min(SIZE, int(round(ch * ps[0] / MM_PER_PX))))\n    nw = max(1, min(SIZE, int(round(cw * ps[1] / MM_PER_PX))))\n    interp = cv2.INTER_AREA if nh < ch else cv2.INTER_LINEAR\n    res = cv2.resize(crop, (nw, nh), interpolation=interp)\n    oy, ox = (SIZE - nh) // 2, (SIZE - nw) // 2\n    out[oy:oy + nh, ox:ox + nw] = res\n    return out\n\n\ndef load_series(sdir, plane_name=None):\n    """Returns (uint8 volume [n, SIZE, SIZE], info dict). Unreadable slices are skipped."""\n    slices, bad = [], 0\n    for fn in os.listdir(sdir):\n        try:\n            slices.extend(_read(os.path.join(sdir, fn)))\n        except Exception:\n            bad += 1\n    info = {"n_raw": len(slices), "n_bad": bad}\n    if not slices:\n        return np.zeros((1, SIZE, SIZE), np.uint8), info\n\n    iop = next((s[1] for s in slices if s[1] is not None), None)\n    if iop is not None:\n        normal = np.cross(iop[:3], iop[3:])\n        plane = int(np.abs(normal).argmax())\n    else:\n        normal, plane = None, PLANES.index(plane_name) if plane_name in PLANES else 0\n    info["plane"] = PLANES[plane]\n\n    if normal is not None and all(s[2] is not None for s in slices):\n        sgn = CANON[plane][0] * np.sign(normal[plane])\n        slices.sort(key=lambda s: float(np.dot(s[2], normal)) * sgn)\n        pos = [float(np.dot(s[2], normal)) for s in slices]\n        info["slice_gap"] = float(np.median(np.abs(np.diff(pos)))) if len(pos) > 1 else 0.0\n    else:\n        slices.sort(key=lambda s: s[4])\n    if len(slices) > MAX_SLICES:\n        keep = np.linspace(0, len(slices) - 1, MAX_SLICES).round().astype(int)\n        slices = [slices[i] for i in keep]\n\n    vol = np.stack([_resample(*_canon_slice(s[0], s[1], s[3], plane)) for s in slices])\n    lo, hi = np.percentile(vol[:, ::4, ::4], [1, 99.5])\n    vol = np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1)\n    info["n"] = len(vol)\n    return (vol * 255).round().astype(np.uint8), info\n\n\ndef _job(args):\n    split, study, series, plane, out_dir = args\n    out = f"{out_dir}/{series}.npy"\n    info = {"study": study, "series": series}\n    if os.path.exists(out):\n        return None\n    try:\n        vol, extra = load_series(f"{DATA}/{split}_series/{study}/{series}", plane)\n        np.save(out + ".tmp.npy", vol)\n        os.replace(out + ".tmp.npy", out)\n        info.update(extra)\n    except Exception as e:\n        info["error"] = f"{type(e).__name__}: {e}"[:200]\n    return info\n\n\ndef main():\n    split = sys.argv[1] if len(sys.argv) > 1 else "train"\n    limit = int(sys.argv[2]) if len(sys.argv) > 2 else None\n    out_dir = f"{WORK}/cache{SIZE}_{split}"\n    os.makedirs(out_dir, exist_ok=True)\n    se = pd.read_csv(f"{DATA}/{split}_series.csv")\n    if limit:\n        se = se.sample(limit, random_state=0)\n    tasks = [(split, a, b, p, out_dir) for a, b, p in zip(se.StudyInstanceUID, se.SeriesInstanceUID, se.Anatomical_Plane)]\n    rows = []\n    with Pool(18) as p:\n        for i, r in enumerate(p.imap_unordered(_job, tasks, chunksize=4)):\n            if r:\n                rows.append(r)\n            if i % 1000 == 0:\n                print(f"{i}/{len(tasks)}", flush=True)\n    df = pd.DataFrame(rows)\n    meta = f"{WORK}/cache{SIZE}_{split}_meta.csv"\n    if os.path.exists(meta) and len(df):\n        df = pd.concat([pd.read_csv(meta), df]).drop_duplicates("series", keep="last")\n    if len(df):\n        df.to_csv(meta, index=False)\n        print("done", len(df), "errors:", int(df["error"].notna().sum()) if "error" in df else 0)\n\n\nif __name__ == "__main__":\n    main()', 'audit_public_data.py': '"""CPU-only public-data replay of the exact ConvNeXt series preprocessing.\n\nDiagnostic only: never makes a competition submission or loads a model.\nRun as an embedded Kaggle script with preprocess.py alongside this file.\n"""\nimport concurrent.futures\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport time\nimport traceback\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nfrom preprocess import load_series, MAX_SLICES\n\nPLANES=[\'Sagittal\',\'Coronal\',\'Axial\']\n\ndef inventory(root, split):\n    studies=pd.read_csv(root/f\'{split}.csv\',dtype={\'StudyInstanceUID\':str})\n    series=pd.read_csv(root/f\'{split}_series.csv\',dtype={\'StudyInstanceUID\':str,\'SeriesInstanceUID\':str})\n    rows=[]; slots={}; errors=[]\n    for r in series.itertuples(index=False):\n        path=root/f\'{split}_series\'/r.StudyInstanceUID/r.SeriesInstanceUID\n        try:\n            count=min(len(os.listdir(path)),MAX_SLICES)\n            if count<3:\n                continue\n            slot=PLANES.index(r.Anatomical_Plane)+3*(1-int(r.Fat_Suppression))\n            if not 0<=slot<6:\n                raise ValueError(f\'invalid slot {slot}\')\n            slots.setdefault((r.StudyInstanceUID,slot),[]).append((count,r.SeriesInstanceUID,path))\n        except Exception as exc:\n            errors.append(dict(study=r.StudyInstanceUID,series=r.SeriesInstanceUID,error=repr(exc)))\n    # Stable descending order exactly matches build_study_table.\n    for (study,slot),candidates in slots.items():\n        candidates.sort(key=lambda r:-r[0])\n        count,sr,path=candidates[0]\n        rows.append((split,study,slot,sr,path,count))\n    present={r[1] for r in rows}\n    return rows,dict(studies=len(studies),series=len(series),selected_series=len(rows),\n                    inventory_errors=errors,empty_studies=sorted(set(studies.StudyInstanceUID)-present))\n\ndef inspect_one(row):\n    split,study,slot,sr,path,count=row\n    result=dict(split=split,study=study,slot=slot,series=sr,file_count_capped=count)\n    started=time.monotonic()\n    try:\n        vol,info=load_series(str(path))\n        result.update(info=info,shape=list(vol.shape),dtype=str(vol.dtype))\n        failures=[]\n        if info.get(\'n_bad\',0): failures.append(\'STRICT_N_BAD_ABORT\')\n        if len(vol)<3: failures.append(\'STRICT_SHORT_SERIES_ABORT\')\n        if vol.ndim!=3 or vol.shape[1:]!=(384,384): failures.append(\'INVALID_VOLUME_SHAPE\')\n        result[\'failures\']=failures\n        # Preserve representative original decoder errors discarded by load_series.\n        if info.get(\'n_bad\',0):\n            from preprocess import _read\n            result[\'decode_errors\']=[]\n            for name in os.listdir(path):\n                try: _read(str(path/name))\n                except Exception as exc:\n                    result[\'decode_errors\'].append(dict(file=name,error=repr(exc)))\n                    if len(result[\'decode_errors\'])>=3: break\n    except Exception as exc:\n        result.update(failures=[\'PREPROCESS_EXCEPTION\'],error=repr(exc),traceback=traceback.format_exc())\n    result[\'seconds\']=time.monotonic()-started\n    return result\n\ndef main():\n    cv2.setNumThreads(1)\n    root=Path(\'/kaggle/input/competitions/rsna-knee-abnormality-detection\')\n    if not root.exists(): root=Path(\'/kaggle/input/rsna-knee-abnormality-detection\')\n    out=Path(\'/kaggle/working\'); started=time.monotonic()\n    tasks=[]; inv={}\n    for split in [\'test\',\'train\']:\n        rows,inv[split]=inventory(root,split); tasks.extend(rows)\n    summary=dict(status=\'RUNNING\',scope=\'public train and visible test only\',gpu=False,\n                 preprocessing_sha256=hashlib.sha256(Path(__file__).with_name(\'preprocess.py\').read_bytes()).hexdigest(),\n                 inventory=inv,total_selected=len(tasks),processed=0,failed_series=0,failures_by_type={},examples=[],\n                 complete=False,hidden_root_cause_confirmed=False)\n    def save():\n        summary[\'elapsed_seconds\']=time.monotonic()-started\n        (out/\'public_data_audit.json\').write_text(json.dumps(summary,indent=2))\n    save()\n    # Four bounded tasks at a time; no array retention between series.\n    with (out/\'public_data_series.jsonl\').open(\'w\') as handle:\n        with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:\n            iterator=iter(tasks)\n            pending={pool.submit(inspect_one,r) for r in [next(iterator,None) for _ in range(4)] if r is not None}\n            while pending:\n                done,pending=concurrent.futures.wait(pending,return_when=concurrent.futures.FIRST_COMPLETED)\n                for job in done:\n                    row=job.result(); handle.write(json.dumps(row)+\'\\n\')\n                    summary[\'processed\']+=1\n                    if row[\'failures\']:\n                        summary[\'failed_series\']+=1\n                        for failure in row[\'failures\']:\n                            summary[\'failures_by_type\'][failure]=summary[\'failures_by_type\'].get(failure,0)+1\n                        if len(summary[\'examples\'])<30: summary[\'examples\'].append(row)\n                    next_row=next(iterator,None)\n                    if next_row is not None: pending.add(pool.submit(inspect_one,next_row))\n                if summary[\'processed\']%100<len(done):\n                    handle.flush();save()\n                    print(\'PUBLIC AUDIT\',summary[\'processed\'],\'/\',len(tasks),\'failed\',summary[\'failed_series\'],flush=True)\n    summary.update(status=\'COMPLETE\',complete=True)\n    save();print(\'PUBLIC AUDIT COMPLETE\',json.dumps(summary),flush=True)\n\nif __name__==\'__main__\': main()\n'}.items(): (src/name).write_text(source,encoding='utf-8')
roots={p.parent for pattern in ['datasets/*/*/cnxt_v0_fold0.pt','*/cnxt_v0_fold0.pt'] for p in Path('/kaggle/input').glob(pattern)}
assert len(roots)==1
envdir=Path('/kaggle/working/public_audit_env')
subprocess.run([sys.executable,'-m','pip','install','-q','--no-index','--no-deps','--target',str(envdir),'--find-links',str(next(iter(roots))),'pylibjpeg','pylibjpeg-libjpeg','pylibjpeg-openjpeg'],check=True)
env=dict(os.environ,PYTHONPATH=str(envdir),CUDA_VISIBLE_DEVICES='')
subprocess.run([sys.executable,'-u',str(src/'audit_public_data.py')],env=env,check=True,timeout=6*3600)
